In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/23 17:33:02 WARN Utils: Your hostname, krock-PCPartner resolves to a loopback address: 127.0.1.1; using 10.159.93.156 instead (on interface enx1e5570e87160)
26/09/23 17:33:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 17:33:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [10]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi_t5))

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

print("df_target: ")
df_target.show()
print("df_transaksi (5 baris pertama dari total", df_transaksi.count(), "baris):")
df_transaksi.show(5)

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.
df_target: 


+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



df_transaksi (5 baris pertama dari total 500 baris):


+--------+--------------------+----------+------------+------------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|
+--------+--------------------+----------+------------+------------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|
+--------+--------------------+----------+------------+------------+
only showing top 5 rows



In [13]:
# Ringkas total pendapatan per kota dari df_transaksi, lalu join dengan df_target. 
# Tambahkan kolom pencapaian_persen. Urutkan hasil dari pencapaian tertinggi.

df_pendapatan = df_transaksi.groupBy("kota").agg(
    spark_sum(col("unit_terjual") * col("harga_satuan")).alias("total_pendapatan")
)

df_fusion = df_pendapatan.join(df_target, on="kota", how="inner").withColumn(
    "pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100
)

df_hasil = df_fusion.orderBy(col("pencapaian_persen").desc())
df_hasil.show()

[Stage 11:>                                                         (0 + 1) / 1]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [15]:
# Menggunakan window function, tentukan kategori dengan pendapatan tertinggi
# di setiap kota (top-1 saja, gunakan row_number()).

pendapatan_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum(col("unit_terjual") * col("harga_satuan")).alias("total_pendapatan_kategori")
)

window_spec = Window.partitionBy("kota").orderBy(
    col("total_pendapatan_kategori").desc()
)

df_hasil = (
    pendapatan_kategori_kota.withColumn(
        "rank", row_number().over(window_spec)
    )
    .filter(col("rank") == 1)
    .drop("rank")
)

df_hasil.orderBy("kota").show()

[Stage 17:>                                                         (0 + 1) / 1]

+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



In [18]:
# Daftarkan df_transaksi dan df_target sebagai temporary view, lalu tulis satu kueri SQL 
# (bukan DataFrame API) yang menampilkan: kota, pic_cabang, dan jumlah transaksi (COUNT) 
# di kota tersebut, diurutkan dari jumlah transaksi terbanyak.


df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")


hasil_sql = spark.sql("""
    SELECT t.kota,tg.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

hasil_sql.show()

[Stage 24:===========================================>              (3 + 1) / 4]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



In [ ]:
# Tulis pada markdown cell (minimal 100 kata): berdasarkan hasil bagian A dan B, cabang 
# mana yang berkinerja paling baik dan cabang mana yang paling perlu perhatian manajemen? 
# Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

Berdasarkan hasil analisis data transaksi dan target bulanan cabang pada Bagian A dan B,
dapat disimpulkan bahwa performa dari masing-masing cabang e-commerce sebagai berikut:

Cabang dengan Kinerja Paling Baik:
Cabang Purworejo (PIC: Fitri) mencatatkan kinerja terbaik dengan pencapaian target 
bulanan sebesar 134,81% (total pendapatan Rp40.443.000 dari target Rp30.000.000). 
Tingginya performa Purworejo didorong oleh kontribusi utama dari kategori Rumah Tangga 
yang menghasilkan pendapatan tertinggi di kota tersebut.

Cabang yang Paling Memerlukan Perhatian Manajemen:
Cabang Yogyakarta (PIC: Joko) menjadi cabang yang paling memerlukan evaluasi mendalam
karena hanya berhasil mencapai 58,70% dari target bulanan 
(total pendapatan Rp35.221.000 dari target Rp60.000.000). Meskipun kategori Elektronik 
menjadi penyumbang terbesar di Yogyakarta, total pendapatan secara keseluruhan masih 
belum mampu mengejar target pasarnya.

Manajemen disarankan untuk melakukan alokasi ulang strategi pemasaran dan evaluasi 
penetapan target di Yogyakarta, serta mereplikasi ulang strategi penjualan kategori Rumah 
Tangga Purworejo ke cabang lainnya.

In [19]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
